# Complementos da v2: bootstrap em dois níveis, RQ1 completa e RQ2 com limiar calibrado

Este notebook complementa os notebooks 07 e 08, sem alterá-los. Ele lê os arquivos gerados por eles e calcula:

1. **Bootstrap da v2 em dois níveis.** O notebook da v2 reamostra pares soltos, o que trata os 5.863 pares como independentes. Aqui a reamostragem é por contexto (`Sentence_control`) e por contexto + termo, como na v1.1.
2. **RQ1 com todos os grupos de comparação.** Os mesmos contrastes do notebook 07 (média por contexto, só contextos compartilhados, Holm por modelo), acrescentando **NRMA** e **Controle de comprimento**, que testam se o efeito vem de qualquer palavra a mais. Também traz o intervalo em dois níveis.
3. **RQ2 com os limiares calibrados.** Recalcula os scores de toxicidade das 50 frases do subconjunto e conta as mudanças de decisão nos três limiares. Mostra também o score médio dos controles, porque no uso figurado o controle troca o termo por outra palavra, e no literal só remove o termo.

**Entrada:** `data_bits/BITS_PTBR_v2_corrigido_com_predicoes.csv` (notebook 07), `data_bits/BITS_PTBR_v2_RQ2_literal_figurado_pareado.csv` (notebook 08) e `results/thresholds.json` (notebook 03). Para as conferências, `results/RQ1_contrastes_Holm_IC95.csv` (notebook 07) e `results/BITS_PTBR_v2_literal_figurado_metricas.csv` (notebook 08).

**Saída (em `results`):** `v2_bootstrap_contexto_termo.csv`, `v2_rq1_contrastes_completos.csv`, `v2_rq2_literal_figurado_scores_tox.csv`, `v2_rq2_literal_figurado_fpr5.csv` e `v2_rq2_comparacao_literal_figurado.csv`.

**GPU:** só a parte 3 usa os modelos, e são 100 textos; roda até em CPU, mas com GPU T4 é mais rápido.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

ROOT = "/content/drive/MyDrive/JIS - capacitismo"
DATA = f"{ROOT}/data_bits"
OUT  = f"{ROOT}/results"
SEED = 42
B = 10000

ARQ_V2  = f"{DATA}/BITS_PTBR_v2_corrigido_com_predicoes.csv"
ARQ_RQ2 = f"{DATA}/BITS_PTBR_v2_RQ2_literal_figurado_pareado.csv"

MODELOS = {"BERTweet-PT": "sent_bertweet", "BERTimbau toxic": "tox1", "ToxiGuardrail-PT": "tox2"}
with open(f"{OUT}/thresholds.json") as f:
    thresholds = json.load(f)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)

v2 = pd.read_csv(ARQ_V2, encoding="utf-8-sig")
assert len(v2) == 5863 and v2.Sentence.is_unique
for p in MODELOS.values():
    assert np.allclose(v2[f"{p}_delta"], v2[f"{p}_score_group"] - v2[f"{p}_score_control"])
print("v2:", len(v2), "pares |", v2.Sentence_control.nunique(), "controles |", v2.SubClass.nunique(), "termos")
print(v2.groupby("Class").agg(pares=("pair_id", "size"), termos=("SubClass", "nunique"),
                              controles=("Sentence_control", "nunique")))

Mounted at /content/drive
v2: 5863 pares | 145 controles | 41 termos
                         pares  termos  controles
Class                                            
Clinico                    870       6        145
Contraste_Idade            353       3        118
Contraste_Nacionalidade    435       3        145
Contraste_Religiao         435       3        145
Controle_Comprimento      1015       7        145
Discurso_Social           1450      10        145
NRMA                       725       5        145
Sem_Deficiencia            580       4        145


## 1. Bootstrap da v2 por contexto e por contexto + termo

- **Contexto:** reamostra os 145 controles com reposição. Corrige a pseudo-replicação (muitos pares compartilham o mesmo controle).
- **Contexto + termo:** reamostra também os termos do grupo. O intervalo passa a valer para o grupo, e não só para os termos escolhidos. É bem mais largo, e isso é um resultado legítimo.

Por termo, só existe o nível de contexto. 10 mil reamostragens, semente 42. Leva alguns minutos.

In [ ]:
def matriz(dados, coluna):
    return dados.pivot_table(index="Sentence_control", columns="SubClass", values=coluna, aggfunc="mean")

def boot_grupo(M, dois_niveis, B=B, seed=SEED):
    M = M.to_numpy(float); nc, nt = M.shape
    rng = np.random.default_rng(seed); medias = np.empty(B)
    for b in range(B):
        ci = rng.integers(0, nc, nc)
        ti = rng.integers(0, nt, nt) if (dois_niveis and nt > 1) else np.arange(nt)
        medias[b] = np.nanmean(M[np.ix_(ci, ti)])
    return np.percentile(medias, [2.5, 97.5])

linhas = []
for modelo, p in MODELOS.items():
    col = f"{p}_delta"
    conjuntos = [("Geral", "Todos", "Todos", v2)]
    conjuntos += [("Por grupo", g, "Todos", d) for g, d in v2.groupby("Class")]
    conjuntos += [("Por termo", g, t, d) for (g, t), d in v2.groupby(["Class", "SubClass"])]
    for nivel, grupo, termo, dados in conjuntos:
        M = matriz(dados, col)
        tipos = [("Contexto", False)] if nivel == "Por termo" else [("Contexto", False), ("Contexto + Termo", True)]
        for tipo, dois in tipos:
            li, ls = boot_grupo(M, dois)
            linhas.append({"Modelo": modelo, "Nivel": nivel, "Grupo": grupo, "Termo": termo, "Bootstrap": tipo,
                           "ScoreSense": dados[col].mean(), "IC95_inf": li, "IC95_sup": ls,
                           "n_pares": len(dados), "n_contextos": M.shape[0], "n_termos": M.shape[1]})
boot = pd.DataFrame(linhas)
boot.to_csv(f"{OUT}/v2_bootstrap_contexto_termo.csv", index=False)

tab = boot[boot.Nivel.isin(["Geral", "Por grupo"])].copy()
tab["IC95"] = tab.apply(lambda r: f"[{r.IC95_inf:.3f}; {r.IC95_sup:.3f}]", axis=1)
print(tab.pivot_table(index=["Modelo", "Grupo"], columns="Bootstrap", values="IC95", aggfunc="first")
         .join(tab.groupby(["Modelo", "Grupo"]).ScoreSense.first().round(3)))

                                                  Contexto  Contexto + Termo  ScoreSense
Modelo           Grupo                                                                  
BERTimbau toxic  Clinico                  [-0.036; -0.005]   [-0.073; 0.038]      -0.020
                 Contraste_Idade            [0.019; 0.038]   [-0.019; 0.088]       0.029
                 Contraste_Nacionalidade  [-0.077; -0.050]  [-0.091; -0.034]      -0.063
                 Contraste_Religiao       [-0.025; -0.001]   [-0.040; 0.023]      -0.013
                 Controle_Comprimento       [0.006; 0.038]   [-0.096; 0.192]       0.022
                 Discurso_Social            [0.211; 0.264]    [0.116; 0.360]       0.238
                 NRMA                     [-0.081; -0.051]  [-0.110; -0.013]      -0.066
                 Sem_Deficiencia            [0.144; 0.184]    [0.005; 0.352]       0.165
                 Todos                      [0.048; 0.079]    [0.008; 0.125]       0.064
BERTweet-PT      Clin

## 2. RQ1: contrastes com todos os grupos de comparação

Para cada controle, a média dos deltas de cada grupo; depois a diferença entre os grupos, só nos controles que os dois têm (118 quando envolve Idade). Teste t sobre as diferenças por contexto, com Holm por modelo (12 comparações), igual ao notebook 07. A célula confere as 24 comparações do notebook 07.

Leitura: **Controle de comprimento** e **NRMA** respondem se o efeito aparece com qualquer palavra a mais. Idade, nacionalidade e religião respondem se a deficiência pesa mais que outras identidades sociais. A coluna `IC_contexto_termo_exclui_zero` indica se a diferença se mantém quando os termos também são reamostrados.

In [ ]:
DEFICIENCIA = ["Clinico", "Discurso_Social"]
COMPARACAO = ["Sem_Deficiencia", "Contraste_Idade", "Contraste_Nacionalidade", "Contraste_Religiao",
              "NRMA", "Controle_Comprimento"]

def boot_contraste(MA, MB, B=B, seed=SEED):
    ctx = MA.index.intersection(MB.index)
    A, Bm = MA.loc[ctx].to_numpy(float), MB.loc[ctx].to_numpy(float)
    nc, na, nb = len(ctx), A.shape[1], Bm.shape[1]
    rng = np.random.default_rng(seed)
    d1, d2 = np.empty(B), np.empty(B)
    for b in range(B):
        ci = rng.integers(0, nc, nc)
        d1[b] = np.nanmean(A[ci]) - np.nanmean(Bm[ci])
        ta, tb = rng.integers(0, na, na), rng.integers(0, nb, nb)
        d2[b] = np.nanmean(A[np.ix_(ci, ta)]) - np.nanmean(Bm[np.ix_(ci, tb)])
    return ctx, np.percentile(d1, [2.5, 97.5]), np.percentile(d2, [2.5, 97.5])

linhas = []
for modelo, p in MODELOS.items():
    col = f"{p}_delta"
    for a in DEFICIENCIA:
        MA = matriz(v2[v2.Class == a], col)
        for b in COMPARACAO:
            MB = matriz(v2[v2.Class == b], col)
            ctx, ic1, ic2 = boot_contraste(MA, MB)
            dif = MA.loc[ctx].mean(axis=1) - MB.loc[ctx].mean(axis=1)
            t, pv = stats.ttest_1samp(dif, 0)
            linhas.append({"Modelo": modelo, "Grupo_deficiencia": a, "Grupo_comparacao": b,
                           "n_contextos": len(ctx), "Media_deficiencia": MA.loc[ctx].mean(axis=1).mean(),
                           "Media_comparacao": MB.loc[ctx].mean(axis=1).mean(), "Diferenca": dif.mean(),
                           "t": t, "p_bruto": pv,
                           "IC95_contexto_inf": ic1[0], "IC95_contexto_sup": ic1[1],
                           "IC95_contexto_termo_inf": ic2[0], "IC95_contexto_termo_sup": ic2[1]})
rq1 = pd.DataFrame(linhas)
rq1["p_Holm"] = rq1.groupby("Modelo").p_bruto.transform(lambda s: multipletests(s, method="holm")[1])
rq1["Signif_Holm"] = rq1.p_Holm < 0.05
rq1["IC_contexto_termo_exclui_zero"] = (rq1.IC95_contexto_termo_inf > 0) | (rq1.IC95_contexto_termo_sup < 0)
rq1.to_csv(f"{OUT}/v2_rq1_contrastes_completos.csv", index=False)

try:
    jan = pd.read_csv(f"{OUT}/RQ1_contrastes_Holm_IC95.csv")
    jan["Modelo"] = jan.Modelo.str.replace("BERTimbau Toxic", "BERTimbau toxic")
    m = rq1.merge(jan, on=["Modelo", "Grupo_deficiencia", "Grupo_comparacao"])
    print("conferência com o notebook 07:", len(m), "comparações |",
          "diferenças iguais:", bool(np.allclose(m.Diferenca, m.Diferenca_ScoreSense)), "\n")
except FileNotFoundError:
    print("(arquivo RQ1 do notebook 07 não encontrado; conferência pulada)\n")

print(rq1[["Modelo", "Grupo_deficiencia", "Grupo_comparacao", "n_contextos", "Diferenca",
           "IC95_contexto_inf", "IC95_contexto_sup", "IC95_contexto_termo_inf", "IC95_contexto_termo_sup",
           "p_Holm", "IC_contexto_termo_exclui_zero"]].round(3).to_string(index=False))

conferência com a Janaina: 24 comparações | diferenças iguais: True 

          Modelo Grupo_deficiencia        Grupo_comparacao  n_contextos  Diferenca  IC95_contexto_inf  IC95_contexto_sup  IC95_contexto_termo_inf  IC95_contexto_termo_sup  p_Holm  IC_contexto_termo_exclui_zero
     BERTweet-PT           Clinico         Sem_Deficiencia          145     -0.126             -0.156             -0.099                   -0.185                   -0.071   0.000                           True
     BERTweet-PT           Clinico         Contraste_Idade          118     -0.153             -0.194             -0.116                   -0.215                   -0.098   0.000                           True
     BERTweet-PT           Clinico Contraste_Nacionalidade          145     -0.175             -0.217             -0.136                   -0.232                   -0.123   0.000                           True
     BERTweet-PT           Clinico      Contraste_Religiao          145     -0.137        

## 3. RQ2: scores de toxicidade do subconjunto literal × figurado

Recalcula os scores dos dois classificadores de toxicidade nas 50 frases e nos seus controles, com a mesma função usada na calibração. A célula confere se as médias batem com as do notebook 08.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def score_texts(model_name, kind, texts, batch_size=32, max_length=512):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()
    if kind == "bertimbau":
        off_id = {v: k for k, v in model.config.id2label.items()}["OFFENSIVE"]
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], padding=True, truncation=True,
                  max_length=max_length, return_tensors="pt").to(device)
        logits = model(**enc).logits
        s = torch.softmax(logits, -1)[:, off_id] if kind == "bertimbau" else 1 - torch.sigmoid(logits.squeeze(-1))
        out.append(s.float().cpu().numpy())
    del model; torch.cuda.empty_cache()
    return np.concatenate(out)

rq2 = pd.read_csv(ARQ_RQ2, encoding="utf-8-sig")
assert len(rq2) == 50
textos = pd.unique(pd.concat([rq2.Sentence, rq2.Sentence_control])).tolist()
for p, nome, kind in [("tox1", "dougtrajano/toxic-comment-classification", "bertimbau"),
                      ("tox2", "nicholasKluge/ToxiGuardrailPT", "guardrail")]:
    sc = dict(zip(textos, score_texts(nome, kind, textos)))
    rq2[f"{p}_score_group"] = rq2.Sentence.map(sc)
    rq2[f"{p}_score_control"] = rq2.Sentence_control.map(sc)
    rq2[f"{p}_delta"] = rq2[f"{p}_score_group"] - rq2[f"{p}_score_control"]
rq2.to_csv(f"{OUT}/v2_rq2_literal_figurado_scores_tox.csv", index=False)
print("dispositivo:", device, "| frases:", len(rq2), "| textos distintos:", len(textos))

try:
    jan = pd.read_csv(f"{OUT}/BITS_PTBR_v2_literal_figurado_metricas.csv")
    for modelo, p in [("BERTimbau Toxic", "tox1"), ("ToxiGuardrail-PT", "tox2")]:
        ref = jan[(jan.Modelo == modelo) & (jan.Nivel == "Por termo e tipo")].set_index(["Termo", "tipo_uso"]).ScoreSense
        nosso = rq2.groupby(["termo", "tipo_uso"])[f"{p}_delta"].mean()
        print(modelo, "| ScoreSense igual ao do notebook 08:", bool(np.allclose(nosso.loc[ref.index], ref, atol=1e-3)))
except FileNotFoundError:
    print("(métricas do notebook 08 não encontradas; conferência pulada)")

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

dispositivo: cuda | frases: 50 | textos distintos: 80
BERTimbau Toxic | ScoreSense igual ao da Janaina: True
ToxiGuardrail-PT | ScoreSense igual ao da Janaina: True


## 4. RQ2: métricas com os três limiares e comparação literal × figurado

Para cada termo e tipo de uso: score médio do controle e da frase com o termo, ScoreSense com intervalo, e quantas das 10 frases mudam de decisão em cada limiar. Com 10 frases, reportamos contagens, não taxas.

**Cuidado na leitura:** no uso literal, o controle remove o termo; no figurado, troca por outra palavra ("tomado", "indiferente", "teimoso"). Se o score médio dos controles figurados for maior, parte da diferença literal × figurado vem do controle, e não do uso do termo. O ScoreDev não se aplica aqui: há um único termo por frase-base.

In [ ]:
def boot_media(x, B=B, seed=SEED):
    x = np.asarray(x, float); rng = np.random.default_rng(seed)
    return np.percentile(x[rng.integers(0, len(x), (B, len(x)))].mean(1), [2.5, 97.5])

linhas = []
for p, modelo in [("tox1", "BERTimbau toxic"), ("tox2", "ToxiGuardrail-PT")]:
    for (termo, uso), d in rq2.groupby(["termo", "tipo_uso"], sort=False):
        li, ls = boot_media(d[f"{p}_delta"])
        linha = {"Modelo": modelo, "Termo": termo, "tipo_uso": uso, "n": len(d),
                 "score_controle_medio": d[f"{p}_score_control"].mean(),
                 "score_termo_medio": d[f"{p}_score_group"].mean(),
                 "ScoreSense": d[f"{p}_delta"].mean(), "IC95_inf": li, "IC95_sup": ls}
        for lim, v in thresholds[p].items():
            c, g = d[f"{p}_score_control"] >= v, d[f"{p}_score_group"] >= v
            linha[f"para_toxico_{lim}"] = int((~c & g).sum())
            linha[f"para_nao_toxico_{lim}"] = int((c & ~g).sum())
        linhas.append(linha)
rq2_tab = pd.DataFrame(linhas)
rq2_tab.to_csv(f"{OUT}/v2_rq2_literal_figurado_fpr5.csv", index=False)
print("Contagens de mudança de decisão (de 10 frases por linha):\n")
print(rq2_tab.round(3).to_string(index=False))

comp = []
for p, modelo in [("tox1", "BERTimbau toxic"), ("tox2", "ToxiGuardrail-PT")]:
    for termo in ["cego", "surdo"]:
        lit = rq2[(rq2.termo == termo) & (rq2.tipo_uso == "literal")][f"{p}_delta"].to_numpy()
        fig = rq2[(rq2.termo == termo) & (rq2.tipo_uso == "figurado")][f"{p}_delta"].to_numpy()
        rng = np.random.default_rng(SEED)
        dif = lit[rng.integers(0, len(lit), (B, len(lit)))].mean(1) - fig[rng.integers(0, len(fig), (B, len(fig)))].mean(1)
        comp.append({"Modelo": modelo, "Termo": termo, "literal": lit.mean(), "figurado": fig.mean(),
                     "literal_menos_figurado": lit.mean() - fig.mean(),
                     "IC95_inf": np.percentile(dif, 2.5), "IC95_sup": np.percentile(dif, 97.5)})
comp = pd.DataFrame(comp)
comp.to_csv(f"{OUT}/v2_rq2_comparacao_literal_figurado.csv", index=False)
print("\nLiteral menos figurado:\n"); print(comp.round(3).to_string(index=False))

Contagens de mudança de decisão (de 10 frases por linha):

          Modelo      Termo tipo_uso  n  score_controle_medio  score_termo_medio  ScoreSense  IC95_inf  IC95_sup  para_toxico_padrao_0.5  para_nao_toxico_padrao_0.5  para_toxico_youden  para_nao_toxico_youden  para_toxico_fpr_5  para_nao_toxico_fpr_5
 BERTimbau toxic       cego  literal 10                 0.117              0.870       0.753     0.723     0.781                      10                           0                   9                       0                  8                      0
 BERTimbau toxic      surdo  literal 10                 0.117              0.728       0.611     0.526     0.696                      10                           0                   4                       0                  2                      0
 BERTimbau toxic cadeirante  literal 10                 0.117              0.363       0.246     0.136     0.387                       1                           0                   1    

## Leitura dos resultados

Critério usado: uma diferença é considerada robusta quando é significativa com Holm **e** o intervalo de 95% em dois níveis (contexto + termo) não inclui o zero. O intervalo só por contexto vale para os termos usados; o de dois níveis vale para o grupo.

### 1. Bootstrap da v2 (ScoreSense por grupo, IC95% contexto + termo)

- **BERTimbau toxic:** só o Discurso Social fica claramente acima de zero (0,238 [0,116; 0,360]). O Sem Deficiência fica no limite (0,165 [0,005; 0,352]). Clínico (−0,020 [−0,073; 0,038]), Idade, Religião e Controle de comprimento (0,022 [−0,096; 0,192]) incluem o zero; NRMA (−0,066) e Nacionalidade (−0,063) ficam abaixo de zero.
- **ToxiGuardrail-PT:** quase todos os grupos ficam acima de zero, e o maior efeito não é de deficiência: Religião 0,293 [0,133; 0,531], Controle de comprimento 0,168 [0,082; 0,264], Discurso Social 0,147 [0,090; 0,212], Nacionalidade 0,131 [0,047; 0,214]. Clínico (0,026) e NRMA (0,024) incluem o zero.
- **BERTweet-PT:** os dois grupos de deficiência são os únicos negativos: Clínico −0,110 [−0,157; −0,065] e Discurso Social −0,071 [−0,129; −0,022]. NRMA (0,151) e Nacionalidade (0,065) são positivos; os demais incluem o zero.
- Os intervalos em dois níveis são bem mais largos que os por par do notebook da v2 (no Discurso Social do BERTimbau, [0,224; 0,252] por par contra [0,116; 0,360] em dois níveis). A variação entre termos de um mesmo grupo é a maior fonte de incerteza.

### 2. RQ1: o efeito é específico de deficiência?

A resposta depende do modelo.

- **BERTweet-PT: sim.** Discurso Social e Clínico têm sentimento mais negativo que todos os seis grupos de comparação, e as 12 diferenças são robustas (por exemplo, Discurso Social − NRMA: −0,222 [−0,408; −0,091]; Discurso Social − Controle de comprimento: −0,085 [−0,156; −0,022]).
- **BERTimbau toxic: sim para o Discurso Social, não para o Clínico.** O Discurso Social fica acima de Idade (+0,220), Nacionalidade (+0,301), Religião (+0,251), NRMA (+0,303) e Controle de comprimento (+0,216 [0,010; 0,395]). Não se distingue do Sem Deficiência em dois níveis (+0,073 [−0,142; 0,273]), porque "neurotípico" e "não deficiente" também aumentam o score. O efeito é do vocabulário associado à deficiência, inclusive quando nega a deficiência. O Clínico não se distingue dos grupos de contraste e fica abaixo do Sem Deficiência (−0,185 [−0,370; −0,020]).
- **ToxiGuardrail-PT: não.** O Discurso Social fica acima de NRMA (+0,123), Sem Deficiência (+0,094) e Idade (+0,155), mas não se distingue do Controle de comprimento (−0,020 [−0,121; 0,077]) nem da Nacionalidade (Holm p = 0,052), e fica abaixo da Religião no intervalo por contexto (−0,146). O modelo reage a menções de características da pessoa em geral, com os maiores deltas em "judeu" (0,545), "careca" (0,362) e "ruivo" (0,271).

**Ressalva sobre o Controle de comprimento:** "careca", que tem uso pejorativo, tem delta de 0,508 no BERTimbau e 0,362 no ToxiGuardrail-PT. Sem ele, a média do grupo cai de 0,022 para −0,059 (BERTimbau) e de 0,168 para 0,135 (ToxiGuardrail-PT). O grupo não é inteiramente neutro, e isso deve ser declarado (análise de sensibilidade sem "careca" a documentar em célula própria).

### 3. RQ2: literal × figurado (BERTimbau toxic)

| Termo e uso | Score do controle | Score com o termo | ScoreSense [IC95%] | Viram tóxicas (0,5 / Youden / 5% FP) |
|---|---|---|---|---|
| cego, literal | 0,117 | 0,870 | 0,753 [0,723; 0,781] | 10 / 9 / 8 de 10 |
| surdo, literal | 0,117 | 0,728 | 0,611 [0,526; 0,696] | 10 / 4 / 2 de 10 |
| cadeirante, literal | 0,117 | 0,363 | 0,246 [0,136; 0,387] | 1 / 1 / 1 de 10 |
| cego, figurado | 0,591 | 0,825 | 0,235 [0,116; 0,360] | 2 / 7 / 5 de 10 |
| surdo, figurado | 0,201 | 0,528 | 0,327 [0,222; 0,430] | 5 / 2 / 1 de 10 |

- Frases literais e neutras com "cego" ("O músico cego ensaiou durante a tarde") recebem score tão alto quanto as figuradas (0,870 contra 0,825). Com o limiar de 5% de falsos positivos, 8 das 10 viram tóxicas. O modelo não distingue o uso do termo: reage à palavra.
- O delta literal é maior que o figurado (cego: +0,518 [0,387; 0,640]; surdo: +0,284 [0,150; 0,419]), mas essa diferença vem sobretudo do controle. No figurado, o controle troca o termo por outra palavra ("tomado", "indiferente", "teimoso", "desinformados"), que já recebe score alto (0,591 em cego). Por isso, a comparação deve ser lida pelo score absoluto das frases com o termo, e não pelo delta.
- Nenhuma frase passa de tóxica para não tóxica em nenhum limiar.
- **Conclusão:** os dados não apoiam a metáfora como explicação do comportamento do modelo no momento da classificação. São mais compatíveis com uma associação lexical aprendida no treino, em que o termo funciona como marcador de ofensa independentemente do sentido (ligação com a RQ5 e com a contribuição E; leitura descritiva, não causal).
- **ToxiGuardrail-PT:** praticamente não reage nas 50 frases (deltas entre −0,007 e 0,019, nenhuma mudança de decisão), embora reaja a "cego" nos templates da v2 (0,141). O efeito desse modelo depende do contexto da frase.
- Limites: 10 frases por célula (contagens, não taxas); frases sintéticas; controles de tipos diferentes (remoção no literal, substituição no figurado). O ScoreDev não se aplica (um único termo por frase-base).

### 4. RQ3: variação entre termos (BERTimbau toxic, ScoreSense por contexto)

- As formas institucionais ficam perto de zero: "pessoa com deficiência" −0,030, "portador de deficiência" −0,008, "sem deficiência" −0,044 (ToxiGuardrail-PT: 0,020, 0,012 e 0,019).
- As formas adjetivas concentram o efeito: "deficiente mental" 0,514, "cego" 0,450, "neurotípico" 0,429, "surdo" 0,395, "deficiente" 0,334, "autista" 0,298, "deficiente físico" 0,288, "cadeirante" 0,157.
- Isso converge com o HateCheck-PT (contribuição D): o modelo marca muito mais as frases com "deficientes" que as com "pessoas com deficiência". A forma de nomear pesa mais que o grupo.